# Mesh Asset Showcase

This notebook visualises the two bundled assets stored in `src/cardiac_latent_ode/assets/`:

| File | Purpose |
|---|---|
| `mesh_remap_bundle.npz` | Maps a *combined* vertex buffer (one flattened array for the whole heart) to four component sub-meshes: **LV endo**, **LV epi**, **RV endo**, **RV epi** |
| `landmark_indices_bundle.npz` | Stores anatomical landmark indices on those component meshes: valve annuli, apex/base rings, and GLS strain arc vertices |

The example mesh `example_mesh.vtk` is the reference geometry whose vertex ordering both bundles assume.
If you rebuild the model for a **different mesh topology**, you would need to recreate both bundles against the new template.

In [ ]:
import io
from importlib.resources import files

import numpy as np
import plotly.graph_objects as go
import pyvista as pv
from plotly.subplots import make_subplots

TEMPLATE_MESH_PATH = "./example_mesh.vtk"

_ASSETS = files("cardiac_latent_ode.assets")

In [ ]:
# ── Template mesh ──────────────────────────────────────────────────────────────
template = pv.read(TEMPLATE_MESH_PATH)
verts = np.array(template.points, dtype=np.float32)           # (5806, 3)
faces = template.faces.reshape(-1, 4)[:, 1:].astype(np.int32) # (11616, 3)

# ── mesh_remap_bundle ──────────────────────────────────────────────────────────
remap = np.load(io.BytesIO(_ASSETS.joinpath("mesh_remap_bundle.npz").read_bytes()), allow_pickle=False)

COMPONENTS = ["lv_endo", "lv_epi", "rv_endo", "rv_epi"]
component_verts = {c: verts[remap[f"map_{c}"]] for c in COMPONENTS}
component_faces = {c: remap[f"faces_{c}"] for c in COMPONENTS}

# ── landmark_indices_bundle ────────────────────────────────────────────────────
lm = np.load(io.BytesIO(_ASSETS.joinpath("landmark_indices_bundle.npz").read_bytes()), allow_pickle=False)

def decode_annulus_loops(flat, offsets):
    """Decode flat+offsets encoding back to list[np.ndarray]."""
    flat_arr = np.asarray(flat, dtype=np.int64)
    off = np.asarray(offsets, dtype=np.int64)
    return [flat_arr[off[i]:off[i+1]] for i in range(len(off) - 1)]

lv_endo_annuli = decode_annulus_loops(lm["lv_endo_annulus_flat"], lm["lv_endo_annulus_offsets"])
lv_epi_annuli  = decode_annulus_loops(lm["lv_epi_annulus_flat"],  lm["lv_epi_annulus_offsets"])
rv_endo_annuli = decode_annulus_loops(lm["rv_endo_annulus_flat"], lm["rv_endo_annulus_offsets"])
rv_epi_annuli  = decode_annulus_loops(lm["rv_epi_annulus_flat"],  lm["rv_epi_annulus_offsets"])

print(f"Template mesh    : {verts.shape[0]:,} vertices, {faces.shape[0]:,} faces")
print(f"Combined n_verts : {int(remap['combined_n_vertices']):,}  (== template vertex count)")

---
## 1  `mesh_remap_bundle.npz`

The combined vertex buffer is the full `example_mesh.vtk` vertex array (N = 5 806).  
Each cardiac component is described by:
- **`map_<name>`** – 1-D integer array mapping *component-local* vertex indices → *global* combined indices
- **`faces_<name>`** – triangle array in *component-local* vertex indexing

At inference time `predict.py` slices `combined_vertices[map_<name>]` to reconstruct each component.

In [ ]:
print(f"{'Component':<12} {'Local vertices':>15} {'Faces':>8} {'Global idx range':>20}")
print("-" * 60)
for c in COMPONENTS:
    m = remap[f"map_{c}"]
    f = remap[f"faces_{c}"]
    print(f"{c:<12} {len(m):>15,} {len(f):>8,} {int(m.min()):>9} – {int(m.max()):<9}")

# Overlap analysis
print()
sets = {c: set(remap[f"map_{c}"].tolist()) for c in COMPONENTS}
print("Shared vertices between components:")
for i, a in enumerate(COMPONENTS):
    for b in COMPONENTS[i+1:]:
        shared = len(sets[a] & sets[b])
        print(f"  {a} ∩ {b}: {shared:,} shared vertices")

In [ ]:
# ── Interactive 3D: 4 components on the template mesh ─────────────────────────
COLORS = {
    "lv_endo": ("#1f77b4", 1.0),   # blue
    "lv_epi":  ("#ff7f0e", 0.25),   # orange
    "rv_endo": ("#2ca02c", 1.0),   # green
    "rv_epi":  ("#d62728", 0.25),   # red
}
LABELS = {
    "lv_endo": "LV endocardium",
    "lv_epi":  "LV epicardium",
    "rv_endo": "RV endocardium",
    "rv_epi":  "RV epicardium",
}

fig = go.Figure()
for c in COMPONENTS:
    v = component_verts[c]
    f = component_faces[c]
    color, opacity = COLORS[c]
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=color,
        opacity=opacity,
        name=LABELS[c],
        showlegend=True,
        lighting=dict(ambient=0.5, diffuse=0.9, roughness=0.4, specular=0.2),
        flatshading=False,
    ))

fig.update_layout(
    title="mesh_remap_bundle: 4 cardiac components on the template mesh",
    scene=dict(
        xaxis_title="x (mm)", yaxis_title="y (mm)", zaxis_title="z (mm)",
        aspectmode="data",
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)),
    ),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=40, b=0),
    width=800, height=600,
)
fig.show()

In [ ]:
# ── Per-component view (2×2 subplots) ─────────────────────────────────────────
from plotly.subplots import make_subplots

fig2 = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}],
           [{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=[LABELS[c] for c in COMPONENTS],
    horizontal_spacing=0.06,
    vertical_spacing=0.08,
)

positions = [(1,1), (1,2), (2,1), (2,2)]
for c, (row, col) in zip(COMPONENTS, positions):
    v = component_verts[c]
    f = component_faces[c]
    color, _ = COLORS[c]
    fig2.add_trace(
        go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=f[:, 0], j=f[:, 1], k=f[:, 2],
            color=color, opacity=1.0,
            lighting=dict(ambient=0.5, diffuse=0.9),
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Label vertex count
    fig2.add_trace(
        go.Scatter3d(
            x=[float(v[:, 0].mean())], y=[float(v[:, 1].mean())], z=[float(v[:, 2].max()) + 5],
            mode="text",
            text=[f"{len(v):,}V / {len(f):,}F"],
            textfont=dict(size=11),
            showlegend=False,
        ),
        row=row, col=col,
    )

fig2.update_layout(
    title="Individual components (V = vertices, F = faces)",
    width=900, height=700,
    margin=dict(l=10, r=10, t=70, b=10),
)
for i in range(1, 5):
    scene_key = f"scene{'' if i == 1 else i}"
    fig2.update_layout(**{scene_key: dict(aspectmode="data")})
fig2.show()

---
## 2  `landmark_indices_bundle.npz`

All indices are in **component-local** vertex space (i.e., index into the array returned by `verts[map_<component>]`).

| Key prefix | Component | Meaning |
|---|---|---|
| `lv_endo_base_idx` | LV endo | 48-vertex base ring |
| `lv_endo_apex_idx` | LV endo | Single apex vertex |
| `lv_endo_annulus_*` | LV endo | 2 closed valve-annulus loops (mitral: 48 pts, aortic: 24 pts) |
| `lv_epi_annulus_*` | LV epi | Corresponding epicardial annulus loops |
| `rv_endo_annulus_*` | RV endo | RV valve annulus loops |
| `rv_epi_annulus_*` | RV epi | RV epicardial annulus loops |
| `lv_gls_2ch_idx` | LV endo | 15-vertex arc for 2-chamber GLS |
| `lv_gls_4ch_idx` | LV endo | 18-vertex arc for 4-chamber GLS |
| `rvfw_gls_4ch_idx` | RV endo | 9-vertex free-wall arc for RV 4CH GLS |

In [ ]:
# ── Summary of all landmark arrays ────────────────────────────────────────────
print(f"{'Key':<35} {'Shape':>10}  Description")
print("-" * 75)
descriptions = {
    "lv_endo_base_idx":        "LV endo base ring (48 vertices)",
    "lv_endo_apex_idx":        "LV endo apex vertex (scalar)",
    "lv_endo_annulus_flat":    "LV endo annulus loops, flattened",
    "lv_endo_annulus_offsets": "LV endo annulus loop offsets",
    "lv_epi_annulus_flat":     "LV epi annulus loops, flattened",
    "lv_epi_annulus_offsets":  "LV epi annulus loop offsets",
    "rv_endo_annulus_flat":    "RV endo annulus loops, flattened",
    "rv_endo_annulus_offsets": "RV endo annulus loop offsets",
    "rv_epi_annulus_flat":     "RV epi annulus loops, flattened",
    "rv_epi_annulus_offsets":  "RV epi annulus loop offsets",
    "lv_gls_2ch_idx":          "LV GLS arc, 2-chamber view",
    "lv_gls_4ch_idx":          "LV GLS arc, 4-chamber view",
    "rvfw_gls_4ch_idx":        "RV free-wall GLS arc, 4-chamber view",
}
for key in sorted(lm.files):
    if key == "format_version":
        continue
    arr = lm[key]
    desc = descriptions.get(key, "")
    shape_str = str(arr.shape) if arr.ndim > 0 else f"scalar = {int(arr)}"
    print(f"  {key:<33} {shape_str:>10}  {desc}")

### 2.1  LV endocardium — base ring, apex, valve annuli

In [ ]:
lv_v = component_verts["lv_endo"]
lv_f = component_faces["lv_endo"]

base_idx  = lm["lv_endo_base_idx"]
apex_idx  = int(lm["lv_endo_apex_idx"])

fig = go.Figure()

# Mesh
fig.add_trace(go.Mesh3d(
    x=lv_v[:, 0], y=lv_v[:, 1], z=lv_v[:, 2],
    i=lv_f[:, 0], j=lv_f[:, 1], k=lv_f[:, 2],
    color="#1f77b4", opacity=0.3, name="LV endo", showlegend=True,
    lighting=dict(ambient=0.6, diffuse=0.8),
))

# Base ring
b = lv_v[base_idx]
# Close the loop for plotting
b_loop = np.vstack([b, b[0]])
fig.add_trace(go.Scatter3d(
    x=b_loop[:, 0], y=b_loop[:, 1], z=b_loop[:, 2],
    mode="lines+markers",
    line=dict(color="gold", width=4),
    marker=dict(size=3, color="gold"),
    name=f"Base ring ({len(base_idx)} pts)",
))

# Apex
fig.add_trace(go.Scatter3d(
    x=[lv_v[apex_idx, 0]], y=[lv_v[apex_idx, 1]], z=[lv_v[apex_idx, 2]],
    mode="markers",
    marker=dict(size=10, color="red", symbol="diamond"),
    name=f"Apex (idx {apex_idx})",
))

# Annulus loops
annulus_colors = ["#ff6b9d", "#c2ff68"]
annulus_names  = ["Mitral annulus (48 pts)", "Aortic annulus (24 pts)"]
for i, loop_idx in enumerate(lv_endo_annuli):
    pts = lv_v[loop_idx]
    pts_closed = np.vstack([pts, pts[0]])
    fig.add_trace(go.Scatter3d(
        x=pts_closed[:, 0], y=pts_closed[:, 1], z=pts_closed[:, 2],
        mode="lines+markers",
        line=dict(color=annulus_colors[i], width=5),
        marker=dict(size=4, color=annulus_colors[i]),
        name=annulus_names[i],
    ))

fig.update_layout(
    title="LV endocardium: base ring, apex, and valve annuli",
    scene=dict(aspectmode="data", xaxis_title="x (mm)", yaxis_title="y (mm)", zaxis_title="z (mm)"),
    legend=dict(x=0.01, y=0.99),
    width=750, height=580, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

### 2.2  LV endocardium — GLS arcs (Global Longitudinal Strain)

GLS is measured along the endocardial wall from base to apex. Two standard echocardiographic views are used:
- **2-chamber (2CH)**: one line from one base annulus point to the apex
- **4-chamber (4CH)**: one line from the opposing base annulus point to the apex

Each arc is an ordered sequence of vertex indices tracing the path.

In [ ]:
gls_2ch = lm["lv_gls_2ch_idx"]
gls_4ch = lm["lv_gls_4ch_idx"]

fig = go.Figure()
fig.add_trace(go.Mesh3d(
    x=lv_v[:, 0], y=lv_v[:, 1], z=lv_v[:, 2],
    i=lv_f[:, 0], j=lv_f[:, 1], k=lv_f[:, 2],
    color="#1f77b4", opacity=0.25, name="LV endo", showlegend=True,
    lighting=dict(ambient=0.6, diffuse=0.8),
))

for idx_arr, label, color in [
    (gls_2ch, f"GLS 2CH ({len(gls_2ch)} pts)", "#e377c2"),
    (gls_4ch, f"GLS 4CH ({len(gls_4ch)} pts)", "#17becf"),
]:
    pts = lv_v[idx_arr]
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="lines+markers",
        line=dict(color=color, width=6),
        marker=dict(size=5, color=color),
        name=label,
    ))
    # Mark endpoints
    fig.add_trace(go.Scatter3d(
        x=[pts[0, 0], pts[-1, 0]], y=[pts[0, 1], pts[-1, 1]], z=[pts[0, 2], pts[-1, 2]],
        mode="markers",
        marker=dict(size=9, color=color, symbol="circle-open", line=dict(width=3)),
        showlegend=False,
    ))

fig.update_layout(
    title="LV endocardium: GLS arcs (longitudinal strain paths)",
    scene=dict(aspectmode="data", xaxis_title="x (mm)", yaxis_title="y (mm)", zaxis_title="z (mm)"),
    legend=dict(x=0.01, y=0.99),
    width=750, height=580, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

### 2.4  RV landmarks — annuli and GLS arc

In [ ]:
rv_v = component_verts["rv_endo"]
rv_f = component_faces["rv_endo"]

fig = go.Figure()
fig.add_trace(go.Mesh3d(
    x=rv_v[:, 0], y=rv_v[:, 1], z=rv_v[:, 2],
    i=rv_f[:, 0], j=rv_f[:, 1], k=rv_f[:, 2],
    color="#2ca02c", opacity=0.25, name="RV endo", showlegend=True,
    lighting=dict(ambient=0.6, diffuse=0.8),
))

# RV endo annulus loops
ann_colors = ["#ff7f0e", "#e377c2"]
ann_names  = ["RV annulus loop 1", "RV annulus loop 2"]
for i, loop_idx in enumerate(rv_endo_annuli):
    pts = rv_v[loop_idx]
    pts_closed = np.vstack([pts, pts[0]])
    fig.add_trace(go.Scatter3d(
        x=pts_closed[:, 0], y=pts_closed[:, 1], z=pts_closed[:, 2],
        mode="lines+markers",
        line=dict(color=ann_colors[i], width=5),
        marker=dict(size=4, color=ann_colors[i]),
        name=f"{ann_names[i]} ({len(loop_idx)} pts)",
    ))

# GLS arcs
for key, label, color in [
    ("rvfw_gls_4ch_idx", "RV free-wall GLS 4CH",  "#bcbd22"),
]:
    pts = rv_v[lm[key]]
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="lines+markers",
        line=dict(color=color, width=6),
        marker=dict(size=5, color=color),
        name=f"{label} ({len(lm[key])} pts)",
    ))

fig.update_layout(
    title="RV endocardium: valve annuli and GLS arc",
    scene=dict(aspectmode="data", xaxis_title="x (mm)", yaxis_title="y (mm)", zaxis_title="z (mm)"),
    legend=dict(x=0.01, y=0.99),
    width=750, height=580, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

### 2.6  Epicardial annuli (LV epi + RV epi)

In [ ]:
fig = go.Figure()

# Show both epi meshes
for c, color, opacity in [("lv_epi", "#ff7f0e", 0.20), ("rv_epi", "#d62728", 0.20)]:
    v = component_verts[c]
    f = component_faces[c]
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=color, opacity=opacity,
        name=LABELS[c], showlegend=True,
        lighting=dict(ambient=0.6, diffuse=0.8),
    ))

lv_epi_v = component_verts["lv_epi"]
rv_epi_v = component_verts["rv_epi"]

for loops, verts_arr, colors, base_name in [
    (lv_epi_annuli,  lv_epi_v, ["#ffbb78", "#98df8a"], "LV epi annulus"),
    (rv_epi_annuli,  rv_epi_v, ["#ff9896", "#aec7e8"], "RV epi annulus"),
]:
    for i, loop_idx in enumerate(loops):
        pts = verts_arr[loop_idx]
        pts_closed = np.vstack([pts, pts[0]])
        fig.add_trace(go.Scatter3d(
            x=pts_closed[:, 0], y=pts_closed[:, 1], z=pts_closed[:, 2],
            mode="lines+markers",
            line=dict(color=colors[i], width=5),
            marker=dict(size=4, color=colors[i]),
            name=f"{base_name} loop {i+1} ({len(loop_idx)} pts)",
        ))

fig.update_layout(
    title="Epicardial annuli: LV epi + RV epi",
    scene=dict(aspectmode="data", xaxis_title="x (mm)", yaxis_title="y (mm)", zaxis_title="z (mm)"),
    legend=dict(x=0.01, y=0.99),
    width=750, height=580, margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()